# Lista de Exercícios 2 - Advanced Econometrics (2026), short version

Mestrado Profissional em Economia, Insper. Prof. Hedibert Freitas Lopes.

**Group:** (names)

**Data.** Mental (Agresti et al. 2026, §4.1.5), $n=40$: $y$ = `impair`, $x_1$ = `life`, $x_2$ = `ses`; $z_j$ = standardized $x_j$. M1–M3: Gaussian errors with regressors $z_1$, $z_2$, $(z_1,z_2)$; M4–M6: the same with $t_\nu(0,\sigma^2)$ errors. Priors: $\beta_0\sim N(25,5^2)$, $\beta_j\sim N(0,3^2)$, $\sigma^2\sim IG(3,50)$, $\nu\sim\text{Gamma}(2,0.1)$.

This version keeps every required step and number of the full solution and cuts discussion and extra checks. Seed, iterations, burn-in and effective sizes are in 2(b) and at the end.

## Setup and data

The data are typed in the file order of page 2 of the statement. The check table compares them with the summaries printed in the statement.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, integrate
from scipy.special import gammaln, logsumexp

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.precision", 3)
SEED = 2026        # master seed: every sampler call below derives its own seed from it
T0 = time.time()

In [ ]:
impair = np.array([17,19,20,20,20,21,21,22,22,23,24,24,25,26,26,26,26,27,27,27,
                   27,28,28,28,28,28,29,30,30,31,31,31,31,32,33,34,34,34,41,41])
life = np.array([46,39,27, 3,10,44,37,35,78,32,33,18,81,22,50,48,45,21,55,45,
                 60,97,37,30,13,40, 5,59,44,35,95,63,42,38,45,70,57,40,49,89])
ses = np.array([84,97,24,85,15,55,78,91,60,74,67,39,87,95,40,52,61,45,88,56,
                70,89,50,90,56,56,40,72,53,38,29,53, 7,32,55,58,16,29, 3,75])
Mental = pd.DataFrame({"impair": impair, "life": life, "ses": ses})
n = len(Mental)
y = impair.astype(float)
z1 = (life - life.mean()) / life.std(ddof=1)
z2 = (ses - ses.mean()) / ses.std(ddof=1)

check = pd.DataFrame({"mean": Mental.mean(), "s.d.": Mental.std(ddof=1),
                      "min": Mental.min(), "max": Mental.max(),
                      "statement mean": [27.30, 44.42, 56.60],
                      "statement s.d.": [5.46, 22.62, 25.28]})
check.round(2)

---
## Question 1: Análise exploratória e correlações

#### Statement 1(a)

> **(a)** Reporte média, mediana, desvio-padrão, mínimo e máximo de $y$, $x_1$ e $x_2$, e faça os histogramas das três variáveis.

In [ ]:
desc = Mental.agg(["mean", "median", "std", "min", "max"]).T.rename(columns={"std": "s.d."})
display(desc.round(2))

labels = {"impair": "y = impair (mental impairment)", "life": "x1 = life (life events)",
          "ses": "x2 = ses (socioeconomic status)"}
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, (c, lab) in zip(ax, labels.items()):
    a.hist(Mental[c], bins=8, edgecolor="white")
    a.axvline(Mental[c].mean(), color="C3", lw=2, label=f"mean {Mental[c].mean():.1f}")
    a.axvline(Mental[c].median(), color="k", ls="--", lw=1.5, label=f"median {Mental[c].median():.0f}")
    a.set_xlabel(lab); a.set_ylabel("count"); a.legend(fontsize=8)
fig.suptitle("Figure 1. Histograms of the three variables (n = 40)")
fig.tight_layout(); plt.show()

### Answer 1(a)

| | mean | median | s.d. | min | max |
|---|---|---|---|---|---|
| $y$ | 27.30 | 27 | 5.46 | 17 | 41 |
| $x_1$ | 44.42 | 43 | 22.62 | 3 | 97 |
| $x_2$ | 56.60 | 56 | 25.28 | 3 | 97 |

- **$y$:** one peak at 26–29 and a mild right skew (mean > median). The two values of 41 (observations 39 and 40) are separated from the rest by an empty bin.
- **$x_1$:** right-skewed, with a long tail up to 97.
- **$x_2$:** spread over the whole scale, with more mass at high values.

#### Statement 1(b)

> **(b)** Faça os diagramas de dispersão $y\times x_1$, $y\times x_2$ e $x_1\times x_2$ (por exemplo, `pairs(Mental)`).

In [ ]:
pairs = [("life", "impair"), ("ses", "impair"), ("life", "ses")]
flag = (Mental.impair <= 17) | (Mental.impair >= 41) | (Mental.life >= 90) | (Mental.ses <= 7)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, (xc, yc) in zip(ax, pairs):
    a.scatter(Mental[xc], Mental[yc], s=18)
    b1, b0 = np.polyfit(Mental[xc], Mental[yc], 1)
    g = np.linspace(Mental[xc].min(), Mental[xc].max(), 2)
    a.plot(g, b0 + b1 * g, color="C3", lw=1.5, label="least-squares line")
    for i in Mental.index[flag]:
        a.annotate(str(i + 1), (Mental[xc][i], Mental[yc][i]), fontsize=7,
                   xytext=(3, 3), textcoords="offset points")
    a.set_xlabel(xc); a.set_ylabel(yc); a.legend(fontsize=8)
fig.suptitle("Figure 2. Scatter plots (numbers = observation index i, file order)")
fig.tight_layout(); plt.show()

### Answer 1(b)

- **$y \times x_1$ and $y \times x_2$:** $y$ rises with $x_1$ and falls with $x_2$. Both relations look linear, with wide scatter.
- **$x_1 \times x_2$:** no clear relation.
- **Points off the trend:** observation 40 ($y=41$ with ses = 75) is far above the $y\times x_2$ line. Observations 22 and 31 (life 97 and 95) are extreme in $x_1$ but close to the line.

#### Statement 1(c)

> **(c)** Calcule as correlações de Pearson $r(y,x_1)$, $r(y,x_2)$ e $r(x_1,x_2)$. Para cada uma, construa um intervalo de confiança de 95% (via transformação $z$ de Fisher) e teste $H_0:\rho=0$. Compare também com a correlação de Spearman.

In [ ]:
def corr_table(df, pairs, level=0.95):
    """Pearson r with Fisher-z CI, t test of H0: rho = 0, and Spearman r_s for each pair."""
    zq = stats.norm.ppf(0.5 + level / 2)
    rows = {}
    for a, b in pairs:
        r = df[a].corr(df[b])
        z, se = np.arctanh(r), 1 / np.sqrt(len(df) - 3)
        t_stat = r * np.sqrt(len(df) - 2) / np.sqrt(1 - r**2)       # ~ t_{n-2} under H0
        rs = stats.spearmanr(df[a], df[b])
        rows[f"r({a}, {b})"] = {
            "Pearson r": r, "CI 2.5%": np.tanh(z - zq * se), "CI 97.5%": np.tanh(z + zq * se),
            "t stat (df=38)": t_stat, "p-value": 2 * stats.t.sf(abs(t_stat), len(df) - 2),
            "Fisher z stat": z / se, "p-value (z)": 2 * stats.norm.sf(abs(z / se)),
            "Spearman r_s": rs.statistic, "p-value (r_s)": rs.pvalue}
    return pd.DataFrame(rows).T

corr = corr_table(Mental, [("impair", "life"), ("impair", "ses"), ("life", "ses")])
corr.round(4)

### Answer 1(c)

CI: $\tanh\!\big(\operatorname{artanh} r \pm 1.96/\sqrt{n-3}\big)$. Test: $t=r\sqrt{n-2}/\sqrt{1-r^2}\sim t_{38}$ under $H_0:\rho=0$.

| pair | $r$ | 95% CI | $t$ | p-value | decision (5%) | Spearman (p-value) |
|---|---|---|---|---|---|---|
| $(y,x_1)$ | 0.372 | [0.069, 0.613] | 2.472 | 0.018 | reject $H_0$ | 0.368 (0.019) |
| $(y,x_2)$ | −0.399 | [−0.632, −0.099] | −2.679 | 0.011 | reject $H_0$ | −0.402 (0.010) |
| $(x_1,x_2)$ | 0.123 | [−0.196, 0.419] | 0.766 | 0.448 | do not reject | 0.088 (0.590) |

Spearman agrees with Pearson for the pairs involving $y$, so the associations are monotone and not driven by extreme values.

#### Statement 1(d)

> **(d)** Interprete sinais e magnitudes. A correlação entre as duas variáveis explicativas é forte o suficiente para gerar preocupação com multicolinearidade? O que você espera, a partir das correlações, sobre os coeficientes do modelo com ambas as variáveis quando comparados aos dos modelos com uma variável só?

In [ ]:
r = Mental.corr()
r12, ry1, ry2 = r.loc["life", "ses"], r.loc["impair", "life"], r.loc["impair", "ses"]
sy = Mental.impair.std(ddof=1)
print(f"VIF = 1/(1 - r12^2) = {1 / (1 - r12**2):.3f}")
# Coefficients in index points per s.d. of x implied by the correlations (least squares):
print(f"one regressor : beta1 = r_y1*s_y = {ry1 * sy:.2f}, beta2 = r_y2*s_y = {ry2 * sy:.2f}")
b1 = (ry1 - ry2 * r12) / (1 - r12**2) * sy
b2 = (ry2 - ry1 * r12) / (1 - r12**2) * sy
print(f"both          : beta1 = {b1:.2f}, beta2 = {b2:.2f}")
print(f"R^2: life only {ry1**2:.3f}, ses only {ry2**2:.3f}, both "
      f"{(ry1**2 + ry2**2 - 2 * ry1 * ry2 * r12) / (1 - r12**2):.3f}")

### Answer 1(d)

- **Signs and magnitudes.** More life events go with more impairment, and higher SES with less. The associations are moderate: each variable explains 14–16% of the variance of $y$, about ±2 points per s.d.
- **Multicollinearity.** None: $r_{12}=0.123$, VIF $=1/(1-r_{12}^2)=1.015$.
- **What to expect with both variables.** $r_{12}>0$ and the two effects have opposite signs, so each one-regressor model is biased toward zero ($r_{y1}=\beta_1^*+\beta_2^*r_{12}$). With both variables the coefficients should grow in absolute value, from 2.03 and −2.18 to 2.34 and −2.46, and $R^2$ should rise to 0.339.

---
## Question 2: Comparação de modelos via fator de Bayes

#### Statement 2(a)

> **(a)** Escreva a verossimilhança dos modelos Gaussianos (M1–M3) e dos modelos Student-t (M4–M6). Para estes últimos, escreva também a verossimilhança aumentada pelas variáveis latentes $\omega_1,\dots,\omega_n$.

### Answer 2(a)

Notation: $x_i$ is the row of the design matrix ($x_i=(1,z_{1i})'$ in M1/M4, $(1,z_{2i})'$ in M2/M5, $(1,z_{1i},z_{2i})'$ in M3/M6), $\beta$ the matching coefficient vector, and $e_i = y_i - x_i'\beta$. The observations are independent given the parameters, so each likelihood is a product.

**Gaussian models M1–M3.** With $y_i\mid\beta,\sigma^2 \sim N(x_i'\beta,\sigma^2)$:
$$L(\beta,\sigma^2;y)=\prod_{i=1}^{n}\frac{1}{\sqrt{2\pi\sigma^2}}\exp\left\{-\frac{(y_i-x_i'\beta)^2}{2\sigma^2}\right\}=(2\pi\sigma^2)^{-n/2}\exp\left\{-\frac{(y-X\beta)'(y-X\beta)}{2\sigma^2}\right\}.$$

**Student-t models M4–M6.** With $y_i\mid\beta,\sigma^2,\nu \sim t_\nu(x_i'\beta,\sigma^2)$ (location $x_i'\beta$, scale $\sigma$):
$$L(\beta,\sigma^2,\nu;y)=\prod_{i=1}^{n}\frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\Gamma\!\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left[1+\frac{(y_i-x_i'\beta)^2}{\nu\sigma^2}\right]^{-\frac{\nu+1}{2}}.$$
**Augmented likelihood (M4–M6).** Use the scale mixture $y_i\mid\beta,\sigma^2,\omega_i\sim N(x_i'\beta,\sigma^2/\omega_i)$, $\omega_i\mid\nu\sim\text{Gamma}(\nu/2,\nu/2)$:
$$L(\beta,\sigma^2,\nu;y,\omega)=\prod_{i=1}^{n}\underbrace{\left(\frac{\omega_i}{2\pi\sigma^2}\right)^{1/2}\exp\left\{-\frac{\omega_i e_i^2}{2\sigma^2}\right\}}_{N(y_i;\,x_i'\beta,\ \sigma^2/\omega_i)}\ \underbrace{\frac{(\nu/2)^{\nu/2}}{\Gamma(\nu/2)}\,\omega_i^{\nu/2-1}e^{-\nu\omega_i/2}}_{\text{Gamma}(\omega_i;\,\nu/2,\,\nu/2)}$$
$$=(2\pi\sigma^2)^{-n/2}\left[\frac{(\nu/2)^{\nu/2}}{\Gamma(\nu/2)}\right]^{n}\prod_{i=1}^{n}\omega_i^{\frac{\nu+1}{2}-1}\exp\left\{-\sum_{i=1}^{n}\omega_i\left(\frac{e_i^2}{2\sigma^2}+\frac{\nu}{2}\right)\right\}.$$

**Check that integrating out $\omega_i$ gives back the t.** Write $u_i=e_i/\sigma$. The $\omega_i$ factor is a Gamma kernel, $\int_0^\infty \omega^{\frac{\nu+1}{2}-1}e^{-\omega(\nu+u_i^2)/2}d\omega=\Gamma\!\left(\tfrac{\nu+1}{2}\right)\left(\tfrac{\nu+u_i^2}{2}\right)^{-\frac{\nu+1}{2}}$. Then
$$\int p(y_i,\omega_i\mid\cdot)\,d\omega_i=\frac{(\nu/2)^{\nu/2}\,\Gamma\!\left(\frac{\nu+1}{2}\right)}{\sqrt{2\pi\sigma^2}\,\Gamma(\nu/2)}\left(\frac{\nu+u_i^2}{2}\right)^{-\frac{\nu+1}{2}}=\frac{\Gamma\!\left(\frac{\nu+1}{2}\right)}{\Gamma\!\left(\frac{\nu}{2}\right)\sqrt{\nu\pi\sigma^2}}\left(1+\frac{u_i^2}{\nu}\right)^{-\frac{\nu+1}{2}},$$
which is the $t_\nu(x_i'\beta,\sigma^2)$ density above. Given $\omega$, the model is a weighted Gaussian regression with weights $\omega_i$. That is what makes Gibbs sampling possible in 2(b).

#### Statement 2(b)

> **(b)** Obtenha amostras da distribuição a posteriori de cada um dos seis modelos via MCMC (por exemplo, Gibbs para M1–M3; Gibbs com os $\omega_i$ e um passo Metropolis–Hastings para $\nu$ em M4–M6; ou Stan/brms). Reporte média, desvio-padrão e intervalo de credibilidade de 95% de todos os parâmetros, e verifique a convergência das cadeias (traços, autocorrelações, tamanho efetivo).

### Answer 2(b): sampler

Priors: $\beta\sim N(m_0,V_0)$, $\sigma^2\sim IG(a,b)$ with $a=3$, $b=50$, $\nu\sim\text{Gamma}(2,0.1)$. Full conditionals:
- $\beta\mid\cdot\sim N\!\big(V_1(V_0^{-1}m_0+X'\Omega y/\sigma^2),\ V_1\big)$, with $V_1=(V_0^{-1}+X'\Omega X/\sigma^2)^{-1}$.
- $\sigma^2\mid\cdot\sim IG\!\big(a+\tfrac n2,\ b+\tfrac12\sum_i\omega_ie_i^2\big)$.
- Gaussian models: $\Omega=I$ in the two lines above.
- t models: $\omega_i\mid\cdot\sim\text{Gamma}\!\big(\tfrac{\nu+1}{2},\tfrac{\nu+e_i^2/\sigma^2}{2}\big)$.
- $\nu$: random-walk Metropolis–Hastings on $\log\nu$ (step 1.2), targeting $p(\nu\mid\beta,\sigma^2,y)\propto p(\nu)\prod_i t_\nu(e_i/\sigma)$ times the Jacobian $\nu$.

Settings: 4 chains started from prior draws, 2,000 burn-in and 10,000 kept per chain, `SEED = 2026`.

In [ ]:
MODELS = {  # name: (regressors, Student-t errors?)
    "M1": (["life"], False), "M2": (["ses"], False), "M3": (["life", "ses"], False),
    "M4": (["life"], True),  "M5": (["ses"], True),  "M6": (["life", "ses"], True),
}
ZCOL = {"life": z1, "ses": z2}
BNAME = {"life": "beta1", "ses": "beta2"}
A0, B0 = 3.0, 50.0        # sigma^2 ~ IG(A0, B0): shape, scale
NU_A, NU_B = 2.0, 0.1     # nu ~ Gamma(NU_A, rate NU_B)


def design(model):
    """Design matrix [1, z_j, ...] and coefficient names for model `model` ('M1'..'M6')."""
    regs, _ = MODELS[model]
    X = np.column_stack([np.ones(n)] + [ZCOL[r] for r in regs])
    return X, ["beta0"] + [BNAME[r] for r in regs]


def beta_prior(k, scale=1.0):
    """Prior means and s.d. of (beta0, beta_j...): N(25, 5^2) and N(0, 3^2), s.d. times `scale`."""
    m0 = np.r_[25.0, np.zeros(k - 1)]
    s0 = np.r_[5.0, np.full(k - 1, 3.0)] * scale
    return m0, s0


def log_t(u, nu):
    """Log density of the standard Student-t with nu d.f. at u (broadcasts)."""
    return (gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(nu * np.pi)
            - (nu + 1) / 2 * np.log1p(u**2 / nu))

In [ ]:
def gibbs(model, mask, n_keep=5000, burn=2000, prior_scale=1.0, seed=1, nu_step=1.2):
    """Batched Gibbs sampler (with one MH step for nu) for model M1..M6.

    mask : (B, n) array of 0/1. Row b says which observations enter chain b. Four rows of
           ones = four independent chains on the full data; row i with a 0 in column i =
           the fit without observation i (exact LOO), so 40 refits run as one batch.
    Returns dict with draws beta (S,B,k), sigma2 (S,B), nu (S,B) or None, omega posterior
    mean (B,n), nu acceptance rate (B,).
    """
    rng = np.random.default_rng(seed)
    X, names = design(model)
    student = MODELS[model][1]
    B, k = mask.shape[0], X.shape[1]
    m0, s0 = beta_prior(k, prior_scale)
    P0, P0m0 = np.diag(1 / s0**2), m0 / s0**2
    n_eff = mask.sum(1)
    # dispersed starting values (drawn from the prior)
    s2 = 1 / rng.gamma(A0, 1 / B0, B)
    nu = rng.gamma(NU_A, 1 / NU_B, B) if student else None
    omega = np.ones((B, n))
    S = n_keep
    out_b, out_s2 = np.empty((S, B, k)), np.empty((S, B))
    out_nu = np.empty((S, B)) if student else None
    om_sum, acc = np.zeros((B, n)), np.zeros(B)
    for it in range(burn + n_keep):
        lam = mask * omega                                   # precision weights
        prec = np.einsum("bi,ij,il->bjl", lam, X, X) / s2[:, None, None] + P0
        rhs = (lam * y) @ X / s2[:, None] + P0m0
        L = np.linalg.cholesky(prec)
        mean = np.linalg.solve(prec, rhs[..., None])[..., 0]
        zz = rng.standard_normal((B, k, 1))
        beta = mean + np.linalg.solve(np.swapaxes(L, 1, 2), zz)[..., 0]   # N(mean, prec^-1)
        e = y - beta @ X.T
        s2 = 1 / rng.gamma(A0 + n_eff / 2, 1 / (B0 + 0.5 * (lam * e**2).sum(1)))
        if student:
            u = e / np.sqrt(s2)[:, None]
            # target: log p(nu | beta, sigma2, y) + log-Jacobian, omega integrated out
            lpost = lambda v: ((mask * log_t(u, v[:, None])).sum(1)
                               + NU_A * np.log(v) - NU_B * v)
            prop = nu * np.exp(nu_step * rng.standard_normal(B))
            ok = np.log(rng.uniform(size=B)) < lpost(prop) - lpost(nu)
            nu = np.where(ok, prop, nu)
            omega = rng.gamma((nu[:, None] + 1) / 2, 2 / (nu[:, None] + u**2))
        if it >= burn:
            g = it - burn
            out_b[g], out_s2[g] = beta, s2
            if student:
                out_nu[g] = nu
                om_sum += omega
                acc += ok
    return {"model": model, "names": names, "beta": out_b, "sigma2": out_s2, "nu": out_nu,
            "omega_mean": om_sum / S if student else None,
            "acc_nu": acc / S if student else None}

In [ ]:
def ess(x):
    """Multi-chain effective sample size (Geyer initial monotone sequence). x: (S, C)."""
    S, C = x.shape
    xc = x - x.mean(0)
    f = np.fft.rfft(xc, n=2 * S, axis=0)
    acov = np.fft.irfft(f * np.conj(f), axis=0)[:S] / S
    W = (acov[0] * S / (S - 1)).mean()
    var_plus = W * (S - 1) / S + (x.mean(0).var(ddof=1) if C > 1 else 0.0)
    rho = 1 - (W - acov.mean(1)) / var_plus
    rho[0] = 1.0
    P = rho[: S - 1 : 2] + rho[1:S:2]
    neg = np.where(P <= 0)[0]
    P = P[: neg[0]] if len(neg) else P
    P = np.minimum.accumulate(P)
    tau = -1 + 2 * P.sum()
    return S * C / max(tau, 1e-12)


def rhat(x):
    """Split R-hat (Gelman et al.). x: (S, C)."""
    h = x.shape[0] // 2
    xs = np.concatenate([x[:h], x[h : 2 * h]], axis=1)
    W = xs.var(0, ddof=1).mean()
    Bv = h * xs.mean(0).var(ddof=1)
    return np.sqrt(((h - 1) / h * W + Bv / h) / W)


def draws_table(fit):
    """Dict param -> (S, C) array of draws (beta's, sigma2, sigma, nu)."""
    d = {nm: fit["beta"][:, :, j] for j, nm in enumerate(fit["names"])}
    d["sigma2"] = fit["sigma2"]
    d["sigma"] = np.sqrt(fit["sigma2"])
    if fit["nu"] is not None:
        d["nu"] = fit["nu"]
    return d


def summarize(fit):
    """Posterior mean, s.d., 95% credible interval, ESS and split R-hat per parameter."""
    rows = {}
    for nm, x in draws_table(fit).items():
        q = np.quantile(x, [0.025, 0.975])
        rows[nm] = {"mean": x.mean(), "sd": x.std(ddof=1), "2.5%": q[0], "97.5%": q[1],
                    "ESS": ess(x), "Rhat": rhat(x)}
    return pd.DataFrame(rows).T

In [ ]:
N_CHAINS, N_KEEP, N_BURN = 4, 10_000, 2_000
full = {}
for k, m in enumerate(MODELS):
    full[m] = gibbs(m, np.ones((N_CHAINS, n)), n_keep=N_KEEP, burn=N_BURN, seed=SEED + 10 * k)
print(f"{N_CHAINS} chains x ({N_BURN} burn-in + {N_KEEP} kept) iterations per model, "
      f"{N_CHAINS * N_KEEP} draws kept; elapsed {time.time() - T0:.0f} s")
for m in ["M4", "M5", "M6"]:
    print(f"{m}: MH acceptance rate for nu by chain = {np.round(full[m]['acc_nu'], 3)}")

In [ ]:
post = pd.concat({m: summarize(full[m]) for m in MODELS}, names=["model", "parameter"])
post.round({"mean": 3, "sd": 3, "2.5%": 3, "97.5%": 3, "ESS": 0, "Rhat": 4})

In [ ]:
def plot_chains(fit, lags=30, thin=25):
    """Trace plots (all chains, thinned for display) and autocorrelations (chain 1)."""
    d = draws_table(fit); d.pop("sigma")
    fig, ax = plt.subplots(len(d), 2, figsize=(11, 1.6 * len(d)), gridspec_kw={"width_ratios": [3, 1]})
    for r_, (nm, x) in enumerate(d.items()):
        for c in range(x.shape[1]):
            ax[r_, 0].plot(np.arange(0, len(x), thin), x[::thin, c], lw=0.4, alpha=0.7)
        ax[r_, 0].set_ylabel(nm)
        xc = x[:, 0] - x[:, 0].mean()
        ac = [1.0] + [np.dot(xc[:-l], xc[l:]) / np.dot(xc, xc) for l in range(1, lags + 1)]
        ax[r_, 1].bar(range(lags + 1), ac, width=0.7)
        ax[r_, 1].set_ylim(-0.1, 1)
    ax[0, 0].set_title(f"{fit['model']}: traces, {x.shape[1]} chains (every {thin}th draw)")
    ax[0, 1].set_title("autocorrelation, chain 1")
    ax[-1, 0].set_xlabel("iteration after burn-in"); ax[-1, 1].set_xlabel("lag")
    fig.tight_layout(); plt.show()

In [ ]:
plot_chains(full["M1"])

In [ ]:
plot_chains(full["M2"])

In [ ]:
plot_chains(full["M3"])

In [ ]:
plot_chains(full["M4"])

In [ ]:
plot_chains(full["M5"])

In [ ]:
plot_chains(full["M6"])

### Answer 2(b)

Posterior mean [95% credible interval]:

| | $\beta_0$ | $\beta_1$ | $\beta_2$ | $\sigma$ | $\nu$ |
|---|---|---|---|---|---|
| M1 | 27.25 [25.67, 28.81] | 1.89 [0.33, 3.45] | – | 5.08 [4.14, 6.30] | – |
| M2 | 27.24 [25.70, 28.78] | – | −2.03 [−3.56, −0.50] | 5.03 [4.09, 6.24] | – |
| M3 | 27.25 [25.82, 28.67] | 2.19 [0.75, 3.61] | −2.31 [−3.71, −0.90] | 4.57 [3.71, 5.69] | – |
| M4 | 27.18 [25.60, 28.76] | 1.82 [0.24, 3.39] | – | 4.80 [3.76, 6.05] | 21.4 [4.6, 57.3] |
| M5 | 27.22 [25.70, 28.72] | – | −2.25 [−3.80, −0.66] | 4.52 [3.39, 5.81] | 16.9 [3.2, 49.4] |
| M6 | 27.27 [25.84, 28.68] | 2.01 [0.57, 3.46] | −2.39 [−3.84, −0.92] | 4.30 [3.34, 5.47] | 20.9 [4.3, 55.4] |

Posterior s.d. are in the table above. All coefficient intervals exclude zero. As expected from 1(d), the coefficients grow in absolute value with both regressors.

**Convergence.** The traces of the four chains overlap with no trend. Autocorrelations are near zero by lag 3 for the $\beta$'s and by about lag 10 for $\nu$. ESS ranges from 7,180 to 40,551 out of 40,000 draws, split $\hat R\le1.001$, and the MH acceptance rate for $\nu$ is 0.51–0.53.

#### Statement 2(c)

> **(c)** Para os modelos M4–M6, compare graficamente a priori e a posteriori de $\nu$. Os dados trazem informação sobre $\nu$? Há evidência de caudas pesadas?

In [ ]:
grid = np.linspace(0.01, 100, 500)
prior_nu = stats.gamma(NU_A, scale=1 / NU_B)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for a, m in zip(ax, ["M4", "M5", "M6"]):
    a.hist(full[m]["nu"].ravel(), bins=np.arange(0, 101, 2.5), density=True, alpha=0.6,
           label="posterior (MCMC)")
    a.plot(grid, prior_nu.pdf(grid), "k", lw=2, label="prior Gamma(2, 0.1)")
    a.set_xlabel(r"$\nu$"); a.set_title(m); a.legend(fontsize=8)
ax[0].set_ylabel("density")
fig.suptitle(r"Figure. Prior and posterior of the degrees of freedom $\nu$")
fig.tight_layout(); plt.show()

nu_tab = {"prior": {"mean": prior_nu.mean(), "median": prior_nu.median(),
                    "2.5%": prior_nu.ppf(0.025), "97.5%": prior_nu.ppf(0.975),
                    "P(nu<10)": prior_nu.cdf(10), "P(nu>30)": prior_nu.sf(30)}}
for m in ["M4", "M5", "M6"]:
    v = full[m]["nu"].ravel()
    nu_tab[m] = {"mean": v.mean(), "median": np.median(v), "2.5%": np.quantile(v, 0.025),
                 "97.5%": np.quantile(v, 0.975), "P(nu<10)": (v < 10).mean(), "P(nu>30)": (v > 30).mean()}
pd.DataFrame(nu_tab).T.round(3)

### Answer 2(c)

The data say little about $\nu$:

| | mean | median | 95% interval | $P(\nu<10)$ |
|---|---|---|---|---|
| prior | 20.0 | 16.8 | [2.4, 55.7] | 0.264 |
| M4 | 21.4 | 17.9 | [4.6, 57.3] | 0.199 |
| M5 | 16.9 | 13.5 | [3.2, 49.4] | 0.350 |
| M6 | 20.9 | 17.7 | [4.3, 55.4] | 0.216 |

- **M4 and M6:** the posterior follows the prior except that $\nu$ below about 4 is ruled out.
- **M5:** a moderate shift toward heavier tails, produced by the omitted regressor (it disappears in M6).
- **Heavy tails?** No evidence: in M4 and M6, $P(\nu<10)$ is lower than under the prior.

#### Statement 2(d)

> **(d)** Estime a verossimilhança marginal (preditiva a priori)
> $$p(y\mid M_k)=\int p(y\mid\theta_k,M_k)\,p(\theta_k\mid M_k)\,d\theta_k,$$
> para $k=1,\dots,6$, por pelo menos um método adequado (por exemplo, amostragem por importância com proposta $t$ multivariada centrada na moda a posteriori, bridge sampling (pacote `bridgesampling`), método de Chib, ou aproximação de Laplace). Evite o estimador da média harmônica e comente por quê. Dica: trabalhe sempre na escala logarítmica.

### Answer 2(d): method

**Importance sampling.**
- Work on $\phi=(\beta,\log\sigma^2[,\log\nu])$, with $\omega$ integrated out (t likelihood), and include the Jacobian.
- Proposal: $q=t_4(\hat\phi, H^{-1})$, centred at the posterior mode, with $H$ the Hessian of $-\log$ posterior.
- Estimate: $\log\hat p(y)=\operatorname{logsumexp}_m\big[\log p(y\mid\phi_m)+\log p(\phi_m)-\log q(\phi_m)\big]-\log M$, with $M=100{,}000$, working on the log scale throughout.

**Why not the harmonic mean** $\big[\frac1S\sum_s 1/p(y\mid\theta^{(s)})\big]^{-1}$:
- $1/p(y\mid\theta)$ is dominated by low-likelihood regions that posterior draws rarely visit. Its variance is typically infinite, so the estimator is unstable and biased upward.
- It is almost insensitive to the prior, while $p(y)$ is not.

In [ ]:
def log_joint(phi, model, prior_scale=1.0):
    """log p(y|theta) + log p(theta) + log Jacobian, on phi = (beta, log sigma2[, log nu]).

    phi : (M, d) array. Omega is integrated out analytically (Student-t likelihood).
    """
    phi = np.atleast_2d(phi)
    X, _ = design(model)
    student = MODELS[model][1]
    k = X.shape[1]
    beta, ls2 = phi[:, :k], phi[:, k]
    s2 = np.exp(ls2)
    e = y - beta @ X.T
    if student:
        lnu = phi[:, k + 1]
        nu = np.exp(lnu)
        ll = log_t(e / np.sqrt(s2)[:, None], nu[:, None]).sum(1) - n / 2 * ls2
        lp = stats.gamma.logpdf(nu, NU_A, scale=1 / NU_B) + lnu
    else:
        ll = -n / 2 * np.log(2 * np.pi * s2) - (e**2).sum(1) / (2 * s2)
        lp = 0.0
    m0, s0 = beta_prior(k, prior_scale)
    lp = lp + stats.norm.logpdf(beta, m0, s0).sum(1)
    lp = lp + stats.invgamma.logpdf(s2, A0, scale=B0) + ls2
    return ll + lp


def to_phi(fit):
    """Posterior draws on the unconstrained scale, pooled over chains: (S*C, d)."""
    cols = [fit["beta"].reshape(-1, fit["beta"].shape[2]), np.log(fit["sigma2"]).reshape(-1, 1)]
    if fit["nu"] is not None:
        cols.append(np.log(fit["nu"]).reshape(-1, 1))
    return np.hstack(cols)


def num_hessian(f, x, h=1e-4):
    """Central-difference Hessian of a scalar function f at x."""
    d = len(x)
    H = np.empty((d, d))
    for i in range(d):
        for j in range(i, d):
            ei, ej = np.eye(d)[i] * h, np.eye(d)[j] * h
            H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej)
                                 + f(x - ei - ej)) / (4 * h * h)
    return H


def mode_and_cov(fit, prior_scale=1.0):
    """Posterior mode on phi and the inverse negative Hessian there."""
    f = lambda p: -log_joint(p, fit["model"], prior_scale)[0]
    res = optimize.minimize(f, to_phi(fit).mean(0), method="BFGS")
    H = num_hessian(f, res.x)
    return res.x, np.linalg.inv(H), -res.fun, H


def logml_is(fit, prior_scale=1.0, M=100_000, df=4, seed=11):
    """Importance-sampling estimate of log p(y|M) with a multivariate-t proposal at the mode.

    Returns (estimate, Monte Carlo s.e. of the estimate, proposal object)."""
    mode, Sig, _, _ = mode_and_cov(fit, prior_scale)
    q = stats.multivariate_t(loc=mode, shape=Sig, df=df, seed=seed)
    th = q.rvs(M)
    lw = log_joint(th, fit["model"], prior_scale) - q.logpdf(th)
    est = logsumexp(lw) - np.log(M)
    w = np.exp(lw - lw.max())
    se = w.std(ddof=1) / np.sqrt(M) / w.mean()
    return est, se, q

In [ ]:
ml = {}
for m in MODELS:
    est, se, _ = logml_is(full[m])
    ml[m] = {"log p(y|M) (IS)": est, "MC s.e.": se}
ml = pd.DataFrame(ml).T
ml.round(3)

### Answer 2(d)

$\log p(y\mid M_k)$: −125.927, −125.481, −122.517, −125.945, −124.743, −122.481 for M1–M6, each with Monte Carlo s.e. ≤ 0.002.

#### Statement 2(e)

> **(e)** Monte uma tabela com $\log p(y\mid M_k)$, os log-fatores de Bayes $\log B_{k*}=\log p(y\mid M_k)-\log p(y\mid M_*)$ em relação ao melhor modelo $M_*$, e as probabilidades a posteriori dos modelos supondo $P(M_k)=1/6$. Interprete usando a escala de Kass & Raftery (1995).

In [ ]:
def kass_raftery(two_log_b):
    """Kass & Raftery (1995) category for 2 log B (evidence for the better model)."""
    v = abs(two_log_b)
    return ("not worth more than a bare mention" if v < 2 else "positive" if v < 6
            else "strong" if v < 10 else "very strong")


def bf_table(logml):
    """log p(y|M_k), log B_k* against the best model, 2 log B, P(M_k|y) with equal prior odds."""
    t_ = pd.DataFrame({"log p(y|M)": logml})
    t_["log B_k*"] = t_["log p(y|M)"] - t_["log p(y|M)"].max()
    t_["2 log B_*k"] = 2 * (t_["log p(y|M)"].max() - t_["log p(y|M)"])
    t_["P(M_k|y)"] = np.exp(t_["log B_k*"]) / np.exp(t_["log B_k*"]).sum()
    t_["K&R (vs best)"] = [("best model" if v == 0 else kass_raftery(v)) for v in t_["2 log B_*k"]]
    return t_

LOGML = ml.iloc[:, 0]          # importance-sampling estimates
bf = bf_table(LOGML)
bf.round(3)

### Answer 2(e)

| | $\log p(y\mid M_k)$ | $\log B_{k*}$ | $2\log B_{*k}$ | $P(M_k\mid y)$ | Kass & Raftery |
|---|---|---|---|---|---|
| M6 | −122.481 | 0 | 0 | 0.458 | best |
| M3 | −122.517 | −0.036 | 0.07 | 0.442 | not worth more than a bare mention |
| M5 | −124.743 | −2.262 | 4.52 | 0.048 | positive |
| M2 | −125.481 | −3.000 | 6.00 | 0.023 | strong (boundary) |
| M1 | −125.927 | −3.446 | 6.89 | 0.015 | strong |
| M4 | −125.945 | −3.463 | 6.93 | 0.014 | strong |

M6 and M3 are tied. Against M6 there is positive evidence against M5 and strong evidence against M1, M2 and M4.

#### Statement 2(f)

> **(f)** Responda: (i) as duas variáveis explicativas são necessárias? (ii) erros Student-t são preferidos aos Gaussianos?

In [ ]:
pairs_bf = [("M3", "M1"), ("M3", "M2"), ("M6", "M4"), ("M6", "M5"),   # (i) both regressors?
            ("M4", "M1"), ("M5", "M2"), ("M6", "M3")]                 # (ii) Student-t vs Gaussian?
pb = pd.DataFrame({f"{a} vs {b}": {"log B": LOGML[a] - LOGML[b], "B": np.exp(LOGML[a] - LOGML[b]),
                                   "2 log B": 2 * (LOGML[a] - LOGML[b]),
                                   "K&R": kass_raftery(2 * (LOGML[a] - LOGML[b]))}
                   for a, b in pairs_bf}).T
display(pb)
P = bf["P(M_k|y)"]
print(f"P(both regressors | y) = P(M3) + P(M6) = {P['M3'] + P['M6']:.3f}")
print(f"P(Student-t errors | y) = P(M4) + P(M5) + P(M6) = {P['M4'] + P['M5'] + P['M6']:.3f}")

### Answer 2(f)

**(i) Both regressors are needed.** $2\log B$ is 6.82 for M3 vs M1 (strong), 5.93 for M3 vs M2 (positive), 6.93 for M6 vs M4 (strong) and 4.52 for M6 vs M5 (positive). $P(M_3\cup M_6\mid y)=0.901$.

**(ii) Student-t errors are not preferred.** $B$ = 1.04 (M6 vs M3), 0.98 (M4 vs M1), 2.09 (M5 vs M2): all not worth more than a bare mention. $P(\text{t}\mid y)=0.520$, close to the prior 0.5, so the simpler Gaussian M3 is the natural choice.

#### Statement 2(g)

> **(g)** Sensibilidade à priori. Repita o item (e) multiplicando por 3 os desvios-padrão a priori de $\beta_0$, $\beta_1$ e $\beta_2$. Os fatores de Bayes mudam? E as posterioris dos coeficientes? Relacione com o paradoxo de Lindley–Bartlett.

In [ ]:
SCALE = 3.0      # prior s.d. of beta0, beta1, beta2 multiplied by 3; sigma2 and nu priors unchanged
full_d, ml_d = {}, {}
for k, m in enumerate(MODELS):
    full_d[m] = gibbs(m, np.ones((N_CHAINS, n)), n_keep=N_KEEP, burn=N_BURN,
                      prior_scale=SCALE, seed=SEED + 300 + 10 * k)
    ml_d[m] = logml_is(full_d[m], prior_scale=SCALE)[0]
bf_d = bf_table(pd.Series(ml_d))
comp = pd.DataFrame({"log p(y|M), base": LOGML, "log p(y|M), diffuse": bf_d["log p(y|M)"],
                     "change": bf_d["log p(y|M)"] - LOGML,
                     "log B_k*, base": bf["log B_k*"], "log B_k*, diffuse": bf_d["log B_k*"],
                     "P(M|y), base": bf["P(M_k|y)"], "P(M|y), diffuse": bf_d["P(M_k|y)"]})
comp.round(3)

In [ ]:
pd_ = pd.concat({"base prior": post[["mean", "sd", "2.5%", "97.5%"]],
                 "diffuse prior (s.d. x3)": pd.concat({m: summarize(full_d[m]) for m in MODELS},
                                                      names=["model", "parameter"])[["mean", "sd", "2.5%", "97.5%"]]},
                axis=1)
pd_.loc[pd_.index.get_level_values(1).str.startswith("beta")].round(3)

In [ ]:
LOGML_D = bf_d["log p(y|M)"]
Pd = bf_d["P(M_k|y)"]
print(f"P(both regressors | y): base {P['M3'] + P['M6']:.3f}, diffuse {Pd['M3'] + Pd['M6']:.3f}")
pd.DataFrame({f"{a} vs {b}": {"2 log B, base": 2 * (LOGML[a] - LOGML[b]),
                              "2 log B, diffuse": 2 * (LOGML_D[a] - LOGML_D[b])}
              for a, b in pairs_bf}).T.round(3)

### Answer 2(g)

- **Bayes factors change.** Every $\log p(y\mid M)$ falls: by about 1.85 in the one-regressor models and 2.62 in M3/M6. Evidence for the second regressor weakens from strong to positive: $2\log B$ goes from 6.82 to 5.34 for M3 vs M1 and from 6.93 to 5.45 for M6 vs M4. $P(M_3\cup M_6\mid y)$ goes from 0.901 to 0.804.
- **Coefficient posteriors barely move.** In M3, $\beta_1$ goes from 2.19 to 2.32 and $\beta_2$ from −2.31 to −2.44, slightly closer to least squares.
- **Why (Lindley–Bartlett).** Widening the prior of a coefficient lowers the prior density at the posterior mode by about $\log 3$, and the likelihood cannot compensate. The cost of $\beta_0$ is shared by all models. Each extra coefficient costs 0.74 to 0.83 nat. As the prior variance grows without bound, the Bayes factor favours the smaller model whatever the data. Bayes factors need proper, carefully elicited priors on the tested parameters.

---
## Question 3: Comparação de modelos via validação cruzada leave-one-out

#### Statement 3 (common part)

> Para cada observação $i=1,\dots,40$ e para cada modelo $M_k$, ajuste o modelo com as prioris do item 2 usando apenas os dados sem a observação $i$, $y_{(-i)}$, e obtenha a distribuição preditiva a posteriori de $y_i$:
> $$p(y_i\mid y_{(-i)},M_k)=\int p(y_i\mid\theta_k,M_k)\,p(\theta_k\mid y_{(-i)},M_k)\,d\theta_k\approx\frac1G\sum_{g=1}^{G}p\big(y_i\mid\theta_k^{(g)},M_k\big),$$
> em que $\theta_k^{(1)},\dots,\theta_k^{(G)}$ são amostras de $p(\theta_k\mid y_{(-i)},M_k)$. Isso requer $6\times40=240$ ajustes (pense em como reaproveitar código e paralelizar).

### Answer 3: method

- **240 fits as 6 runs.** Each model runs once with 40 chains, where chain $i$ leaves out observation $i$. This gives the exact $p(\theta\mid y_{(-i)})$; each chain keeps 5,000 draws after a 2,000-iteration burn-in.
- **Log predictive density.** $\log p(y_i\mid y_{(-i)})=\operatorname{logsumexp}_g\log p(y_i\mid\theta^{(g)})-\log G$. For the t models, $p(y_i\mid\theta)$ is the $t_\nu(x_i'\beta,\sigma^2)$ density.
- **Point forecasts.** The predictive mean is $\bar\mu_i$. The predictive median solves $\frac1G\sum_gF\big((m-\mu_i^{(g)})/\sigma^{(g)}\big)=\frac12$.

In [ ]:
def loo_exact(fit):
    """From a batched LOO fit (chain i omits obs i): predictive mean, median, log density.

    Also returns the Monte Carlo s.e. of each log predictive density (delta method,
    treating the draws as independent)."""
    X, _ = design(fit["model"])
    student = fit["nu"] is not None
    pmean, pmed, lpd, lse = np.empty(n), np.empty(n), np.empty(n), np.empty(n)
    for i in range(n):
        mu = fit["beta"][:, i, :] @ X[i]
        sd = np.sqrt(fit["sigma2"][:, i])
        if student:
            nu = fit["nu"][:, i]
            lp = log_t((y[i] - mu) / sd, nu) - np.log(sd)
            cdf = lambda v: stats.t.cdf((v - mu) / sd, nu).mean() - 0.5
        else:
            lp = stats.norm.logpdf(y[i], mu, sd)
            cdf = lambda v: stats.norm.cdf((v - mu) / sd).mean() - 0.5
        lpd[i] = logsumexp(lp) - np.log(len(lp))
        p = np.exp(lp - lp.max())
        lse[i] = p.std(ddof=1) / np.sqrt(len(p)) / p.mean()
        pmean[i] = mu.mean()
        pmed[i] = optimize.brentq(cdf, mu.min() - 50, mu.max() + 50, xtol=1e-6)
    return pd.DataFrame({"y": y, "pred_mean": pmean, "pred_median": pmed, "lpd": lpd, "lpd_se": lse})

In [ ]:
LOO_KEEP, LOO_BURN = 5_000, 2_000
def run_loo(prior_scale=1.0, seed0=SEED + 500):
    """Exact LOO for all six models: one batched run of 40 chains per model (chain i omits obs i)."""
    out = {}
    for k, m in enumerate(MODELS):
        f = gibbs(m, 1 - np.eye(n), n_keep=LOO_KEEP, burn=LOO_BURN, prior_scale=prior_scale,
                  seed=seed0 + 10 * k)
        out[m] = loo_exact(f)
    return out

loo = run_loo()
print(f"6 models x 40 leave-one-out fits, {LOO_BURN} burn-in + {LOO_KEEP} kept each; "
      f"elapsed {time.time() - T0:.0f} s")

#### Statement 3(a)

> **(a)** Para cada modelo, calcule os três critérios abaixo, avaliados na observação deixada de fora:
> - Erro quadrático médio: $\text{MSE}_k=\frac1n\sum_i(y_i-\hat y_{i,k})^2$, com $\hat y_{i,k}=E(y_i\mid y_{(-i)},M_k)$;
> - Erro absoluto médio: $\text{MAE}_k=\frac1n\sum_i|y_i-\tilde y_{i,k}|$, com $\tilde y_{i,k}$ a mediana de $p(y_i\mid y_{(-i)},M_k)$;
> - Escore logarítmico preditivo: $\text{LPD}_k=\sum_i\log p(y_i\mid y_{(-i)},M_k)$ (quanto maior, melhor).
>
> Por que faz sentido usar a média preditiva no MSE e a mediana preditiva no MAE?

#### Statement 3(b)

> **(b)** Apresente uma tabela com MSE, MAE e LPD dos seis modelos e o ordenamento dos modelos segundo cada critério. Os três critérios concordam entre si?

In [ ]:
def criteria(loo):
    """MSE (predictive mean), MAE (predictive median), LPD and its MC s.e., with ranks."""
    t_ = pd.DataFrame({m: {"MSE": ((d.y - d.pred_mean) ** 2).mean(),
                           "MAE": (d.y - d.pred_median).abs().mean(),
                           "LPD": d.lpd.sum(), "LPD MC s.e.": np.sqrt((d.lpd_se ** 2).sum())}
                       for m, d in loo.items()}).T
    t_["rank MSE"] = t_["MSE"].rank().astype(int)
    t_["rank MAE"] = t_["MAE"].rank().astype(int)
    t_["rank LPD"] = (-t_["LPD"]).rank().astype(int)
    return t_

crit = criteria(loo)
crit.round(3)

### Answer 3(a)

The predictive mean minimizes expected squared error, and the median minimizes expected absolute error, under the predictive distribution. Each criterion therefore scores the point forecast that is optimal for its own loss.

### Answer 3(b)

| | MSE | MAE | LPD | ranks (MSE, MAE, LPD) |
|---|---|---|---|---|
| M1 | 27.774 | 4.264 | −124.190 | 6, 6, 4 |
| M2 | 27.656 | 3.946 | −124.321 | 4, 3, 6 |
| M3 | 23.263 | 3.804 | −120.358 | 2, 2, 2 |
| M4 | 27.772 | 4.264 | −124.288 | 5, 5, 5 |
| M5 | 27.463 | 3.960 | −123.445 | 3, 4, 3 |
| M6 | 23.167 | 3.792 | −120.290 | 1, 1, 1 |

- **Agreement.** The criteria agree on the top: M6 ≈ M3. The gap between them is 0.068 in LPD, close to its Monte Carlo error, so there is no real difference. Both are well ahead of the rest (MSE 16% lower).
- **Disagreement.** The criteria disagree only on the order of the one-regressor models. LPD scores the whole predictive density, MSE and MAE only its centre.

#### Statement 3(c)

> **(c)** Faça um gráfico das contribuições individuais $\log p(y_i\mid y_{(-i)},M_6)-\log p(y_i\mid y_{(-i)},M_3)$ contra $i$ (ou contra $y_i$). Quais observações mais favorecem os erros Student-t? Elas aparecem como atípicas nos gráficos do item 1?

In [ ]:
idx = np.arange(1, n + 1)
d63 = pd.Series((loo["M6"].lpd - loo["M3"].lpd).values, index=idx)
top = d63.sort_values(ascending=False).index[:5]     # observation numbers (1-based)
low = d63.sort_values().index[:5]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].vlines(idx, 0, d63.values, lw=2); ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_xlabel("observation i (file order)")
ax[1].scatter(y, d63.values, s=18); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel(r"$y_i$")
for a, xv in zip(ax, [idx, y]):
    a.set_ylabel(r"$\log p(y_i|y_{(-i)},M_6) - \log p(y_i|y_{(-i)},M_3)$")
    for i in list(top) + list(low):
        a.annotate(str(i), (xv[i - 1], d63[i]), fontsize=8, xytext=(3, 2), textcoords="offset points")
fig.suptitle("Figure. Pointwise LOO log-score difference, Student-t (M6) minus Gaussian (M3)")
fig.tight_layout(); plt.show()

# LOO residual: y_i minus the M3 predictive mean computed without observation i
loo_res = (loo["M3"].y - loo["M3"].pred_mean).values
tab = pd.DataFrame({"y": y, "life": life, "ses": ses, "LOO residual (M3)": loo_res,
                    "LOO residual / sigma": loo_res / post.loc[("M3", "sigma"), "mean"],
                    "diff M6-M3": d63.values}, index=idx)
print(f"sum of differences = LPD(M6) - LPD(M3) = {d63.sum():.3f}; "
      f"{(d63 > 0).sum()} observations favour M6, {(d63 < 0).sum()} favour M3")
print(f"obs. 22 and 31 (life = 97 and 95, beyond the fence in 1(a)): diff = {d63[22]:.3f} and {d63[31]:.3f}")
pd.concat({"most favour M6": tab.loc[top], "most favour M3": tab.loc[low]}).round(3)

### Answer 3(c)

- **Who favours the t errors.** Observations 40 (+0.198) and 39 (+0.087) favour them most. Both have $y=41$, the largest LOO residuals under M3 (2.8$\sigma$ and 2.1$\sigma$). The total advantage is only 0.068.
- **Who favours M3.** Observations 5 and 3 ($y=20$) favour M3 most.
- **Atypical in item 1?** Observations 39 and 40 are the two isolated top values of the histogram of $y$, and 40 is the point furthest above the $y\times x_2$ line. The extreme life values (22, 31) gain little: leverage is not the same as a large residual.

#### Statement 3(d)

> **(d)** Compare as conclusões da validação cruzada com as do fator de Bayes (item 2). Lembre que $\exp(\text{LPD}_k)$ é o produto das ordenadas preditivas condicionais (CPO) e que $\exp(\text{LPD}_k-\text{LPD}_l)$ é o pseudo-fator de Bayes de Geisser & Eddy (1979). Por que os dois enfoques podem discordar? Qual deles é mais sensível à priori? Repita o cálculo do LPD com as prioris mais difusas do item 2(g) e comente.

In [ ]:
cmp = pd.DataFrame({"log p(y|M) (BF)": LOGML, "rank BF": (-LOGML).rank().astype(int),
                    "LPD (LOO)": crit["LPD"], "rank LPD": crit["rank LPD"]})
display(cmp.round(3))
for a, b in [("M6", "M3"), ("M3", "M1"), ("M3", "M2"), ("M5", "M2")]:
    print(f"{a} vs {b}: log B = {LOGML[a] - LOGML[b]:7.3f}   log pseudo-B = LPD diff = "
          f"{crit.LPD[a] - crit.LPD[b]:7.3f}   (pseudo-B = {np.exp(crit.LPD[a] - crit.LPD[b]):.2f})")

In [ ]:
loo_d = run_loo(prior_scale=SCALE, seed0=SEED + 800)
crit_d = criteria(loo_d)
sens = pd.DataFrame({"LPD, base": crit["LPD"], "LPD, diffuse": crit_d["LPD"],
                     "change LPD": crit_d["LPD"] - crit["LPD"],
                     "change log p(y|M)": comp["change"]})
print(f"elapsed {time.time() - T0:.0f} s")
sens.round(3)

### Answer 3(d)

- **Same conclusions.** Both approaches find that both regressors are needed and that t and Gaussian errors are tied. $\log B_{63}=0.036$; the log pseudo-Bayes factor $\text{LPD}_6-\text{LPD}_3=0.068$ gives a pseudo-$B$ of 1.07. Here $\exp(\text{LPD})=\prod_i\text{CPO}_i$.
- **Only difference.** The order of M1 and M2.
- **Why they can differ.** $p(y\mid M)=\prod_i p(y_i\mid y_{1:i-1})$ includes predictions made from the prior alone, so the Bayes factor scores model plus prior. LOO scores the posterior after 39 observations, where the prior is washed out.
- **Prior sensitivity.** The Bayes factor is far more sensitive: under the diffuse priors $\log p(y\mid M)$ falls by 1.8–2.6, while the LPD changes by at most 0.17 and keeps its ranking.

#### Statement 3(e)

> **(e)** (Para os mais energizados) Em vez de reajustar o modelo 40 vezes, aproxime $p(y_i\mid y_{(-i)},M_k)$ a partir de um único ajuste com todos os dados, usando a identidade $p(y_i\mid y_{(-i)})=\{E[1/p(y_i\mid\theta)\mid y]\}^{-1}$ (Gelfand, Dey & Chang, 1992), ou o PSIS-LOO do pacote `loo` (Vehtari, Gelman & Gabry, 2017). Compare com os valores exatos do item (b) e comente as diferenças observação a observação.

### Answer 3(e): method

**GDC.** $p(y_i\mid y_{(-i)})=\big\{E[1/p(y_i\mid\theta)\mid y]\big\}^{-1}$, estimated with the 40,000 full-data draws.

**PSIS.** Take the same weights $1/p(y_i\mid\theta^{(s)})$, smooth their largest 600 values with a fitted generalized Pareto distribution, and use $\hat k<0.7$ as the reliability check.

In [ ]:
def loglik_matrix(fit):
    """log p(y_i | theta^(g)) for a full-data fit, pooled over chains: (S*C, n)."""
    X, _ = design(fit["model"])
    mu = np.einsum("sck,ik->sci", fit["beta"], X).reshape(-1, n)
    sd = np.sqrt(fit["sigma2"]).reshape(-1, 1)
    if fit["nu"] is None:
        return stats.norm.logpdf(y, mu, sd)
    nu = fit["nu"].reshape(-1, 1)
    return log_t((y - mu) / sd, nu) - np.log(sd)


def gdc_loo(ll):
    """Gelfand-Dey-Chang: log CPO_i = -log E[1/p(y_i|theta) | y]."""
    return -(logsumexp(-ll, axis=0) - np.log(ll.shape[0]))


def gpdfit(x):
    """Generalized Pareto fit (Zhang & Stephens 2009, with the weak prior used in PSIS)."""
    N = len(x)
    m = 30 + int(np.sqrt(N))
    bs = 1 - np.sqrt(m / (np.arange(1, m + 1) - 0.5))
    bs = bs / (3 * x[int(N / 4 + 0.5) - 1]) + 1 / x[-1]
    ks = np.log1p(-bs[:, None] * x).mean(1)
    L = N * (np.log(-bs / ks) - ks - 1)
    w = 1 / np.exp(L[None, :] - L[:, None]).sum(1)   # w_i = 1 / sum_j exp(L_j - L_i)
    b = (bs * w).sum() / w.sum()
    k = np.log1p(-b * x).mean()
    sigma = -k / b
    k = (N * k + 10 * 0.5) / (N + 10)                 # weak prior shrinking k toward 0.5
    return k, sigma


def psis_loo(ll):
    """PSIS-LOO (Vehtari, Gelman & Gabry 2017). ll: (S, n). Returns (elpd_i, khat_i)."""
    S = ll.shape[0]
    M = int(np.ceil(min(0.2 * S, 3 * np.sqrt(S))))
    elpd, khat = np.empty(ll.shape[1]), np.empty(ll.shape[1])
    for i in range(ll.shape[1]):
        lw = -ll[:, i] - (-ll[:, i]).max()
        order = np.argsort(lw)
        cut = lw[order[-M - 1]]
        x = np.exp(lw[order[-M:]]) - np.exp(cut)
        k, sig = gpdfit(x)
        p = (np.arange(M) + 0.5) / M
        q = sig * np.expm1(-k * np.log1p(-p)) / k + np.exp(cut)
        lw[order[-M:]] = np.log(q)
        lw = np.minimum(lw, 0.0)
        lw -= logsumexp(lw)
        elpd[i], khat[i] = logsumexp(lw + ll[:, i]), k
    return elpd, khat

In [ ]:
appr, khat, pw = {}, {}, {}
for m in MODELS:
    ll = loglik_matrix(full[m])          # 40,000 draws x 40 observations, full-data fit
    g = gdc_loo(ll)
    p_, k_ = psis_loo(ll)
    pw[m] = pd.DataFrame({"exact": loo[m].lpd.values, "GDC": g, "PSIS": p_, "khat": k_}, index=idx)
    appr[m] = {"LPD exact": loo[m].lpd.sum(), "LPD GDC": g.sum(), "LPD PSIS": p_.sum(),
               "max |GDC - exact|_i": np.abs(g - loo[m].lpd).max(),
               "max |PSIS - exact|_i": np.abs(p_ - loo[m].lpd).max(), "max khat": k_.max()}
pd.DataFrame(appr).T.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for a, m in zip(ax, ["M3", "M6"]):
    d = pw[m]
    a.plot(idx, d.GDC - d.exact, "o", ms=4, label="GDC - exact")
    a.plot(idx, d.PSIS - d.exact, "x", ms=5, label="PSIS - exact")
    a.axhline(0, color="k", lw=0.8)
    a.set_title(m); a.set_xlabel("observation i"); a.set_ylabel("difference in log p(y_i | y_(-i))")
    a.legend(fontsize=8)
fig.suptitle("Figure. Single-fit approximations minus exact LOO, observation by observation")
fig.tight_layout(); plt.show()
w = pw["M6"].assign(d=lambda t: (t.PSIS - t.exact).abs()).sort_values("d", ascending=False).head(4)
w.round(3)

### Answer 3(e)

GDC and PSIS match the exact LOO:
- The totals are within 0.05 of the exact LPD, and per observation the differences are at most 0.026.
- Every $\hat k\le0.53$, so the estimates are reliable.
- The largest differences are at the high-residual observations (40, 39, 5).

The single-fit approximations give the same ranking with 6 fits instead of 240.

---
## Computational details

In [ ]:
ess_all = pd.concat({m: summarize(full[m])[["ESS", "Rhat"]] for m in MODELS})
print(f"Seed: SEED = {SEED} (each call uses SEED + offset)")
print(f"Full-data fits: {N_CHAINS} chains, burn-in {N_BURN}, {N_KEEP} kept per chain "
      f"({N_CHAINS * N_KEEP} draws), no thinning")
print(f"LOO fits: 240 (x2 priors), burn-in {LOO_BURN}, {LOO_KEEP} kept each")
print(f"ESS range over all parameters: {ess_all.ESS.min():.0f} to {ess_all.ESS.max():.0f}; "
      f"max split R-hat {ess_all.Rhat.max():.4f}")
print(f"Total run time: {time.time() - T0:.0f} s")

### Conclusion

- **Both regressors are needed:** $P(M_3\cup M_6\mid y)=0.90$ and the LOO MSE is 16% lower.
- **Student-t errors are not supported:** $B_{63}=1.04$ and the posterior of $\nu$ is close to its prior. The simpler M3 is preferred.
- **Prior sensitivity:** the Bayes factors depend on the prior (Lindley–Bartlett), while the LOO criteria do not.